# * 2026 : Target Revenue TRUE
    96 Areas

In [35]:
import configparser
import datetime as dt
import pandas as pd
import numpy as np
import oracledb
import re

config = configparser.ConfigParser()
config.read('../../my_config.ini')
config.sections()

TDMDBPR_user = config['TDMDBPR']['username']
TDMDBPR_pwd = config['TDMDBPR']['password']
TDMDBPR_db = config['TDMDBPR']['db']
TDMDBPR_host = config['TDMDBPR']['host']
TDMDBPR_port = config['TDMDBPR']['port']

AKPIPRD_user = config['AKPIPRD']['username']
AKPIPRD_pwd = config['AKPIPRD']['password']
AKPIPRD_db = config['AKPIPRD']['db']
AKPIPRD_host = config['AKPIPRD']['host']
AKPIPRD_port = config['AKPIPRD']['port']

curr_dt = dt.datetime.now().date()
str_curr_dt = curr_dt.strftime('%Y%m%d')
curr_dt

datetime.date(2026, 3, 19)

## ETL Process...

### Step 1 : Import Data Source

In [36]:
''' Rawdata '''

# Target_Revenue_TRUE_Y2026
src_file = '../../data/Target/data/Revenue/Target_Revenue_TRUE_Y2026.xlsx'
src_df_cols = ['CUST_TYPE', 'TM_KEY_MTH', 'METRIC_CD', 'METRIC_NAME', 'METRIC_VALUE', 'COMP_CD', 'VERSION', 'AREA_TYPE', 'AREA_CD', 'AREA_DESC']

src_b2c_sheet = 'B2C Rawdata'
src_b2c_df = pd.read_excel(src_file, sheet_name=src_b2c_sheet, index_col=None) 
src_b2c_df = src_b2c_df.loc[src_b2c_df['AREA_TYPE']=="HH"]
src_b2c_df['CUST_TYPE'] = 'B2C'
src_b2c_df = src_b2c_df[src_df_cols]

src_b2b_sheet = 'B2B Rawdata'
src_b2b_df = pd.read_excel(src_file, sheet_name=src_b2b_sheet, index_col=None) 
src_b2b_df['CUST_TYPE'] = 'B2B'
src_b2b_df = src_b2b_df[src_df_cols]

src_df = pd.concat([src_b2c_df, src_b2b_df])
src_df.rename(columns={'AREA_CD': 'AREA_KEY', 'METRIC_VALUE': 'TARGET_MTH'}, inplace=True)
src_df['COMP_CD'] = 'TRUE' #src_df['COMP_CD'].astype(str)
# src_df = src_df.replace(np.nan, None)
src_df = src_df.reset_index(drop=True)

print(f'\nsrc_df : {src_df.shape[0]} rows, {src_df.shape[1]} columns')
src_df#.tail(3)


src_df : 1155 rows, 10 columns


,CUST_TYPE,TM_KEY_MTH,METRIC_CD,METRIC_NAME,TARGET_MTH,COMP_CD,VERSION,AREA_TYPE,AREA_KEY,AREA_DESC
0,B2C,202601,TB1R000100,Prepaid Revenue : TMH,3.040760e+07,TRUE,T,HH,907030,"BKK : Bang Khen, Lat Phrao, Wang Thonglang"
1,B2C,202601,TB1R000100,Prepaid Revenue : TMH,2.079064e+07,TRUE,T,HH,907016,"BKK : Bang Sue, Chatuchak"
2,B2C,202601,TB1R000100,Prepaid Revenue : TMH,2.664091e+07,TRUE,T,HH,907017,"BKK : Don Mueang, Sai Mai, Lak Si"
3,B2C,202601,TB1R000100,Prepaid Revenue : TMH,4.288086e+07,TRUE,T,HH,907019,"BKK : Lat Krabang, Nong Chok, Khlong Sam Wa"
4,B2C,202601,TB1R000100,Prepaid Revenue : TMH,2.587798e+07,TRUE,T,HH,907020,"BKK : Min Buri, Khan Na Yao, Bueng Kum"
...,...,...,...,...,...,...,...,...,...,...
1150,B2C,202603,TB4R000100,TVS Revenue,6.508133e+05,TRUE,T,HH,906111,TRANG
1151,B2C,202603,TB4R000100,TVS Revenue,2.730801e+05,TRUE,T,HH,906112,YALA
1152,B2B,202601,TB2R020100,Postpaid Revenue B2B : TMH,3.182800e+08,TRUE,T,P,P,Nationwide
1153,B2B,202602,TB2R020100,Postpaid Revenue B2B : TMH,3.182800e+08,TRUE,T,P,P,Nationwide


In [37]:
''' Add Columns '''

def product_group(v_cd):
    cd = v_cd
    result = ''
    if re.search('B1', cd): result = 'Prepaid'
    elif re.search('B2', cd): result = 'Postpaid'
    elif re.search('B3', cd): result = 'TOL'
    elif re.search('B4', cd): result = 'TVS'
    else: result = 'Unknown' 
    return result

src_df['PRODUCT_GRP'] = src_df.apply(lambda x: product_group(x['METRIC_CD']), axis=1)
# src_df['FREQUENCY'] = np.where(src_df['PRODUCT_GRP']=='Prepaid', 'DAY', 'BILL')
src_df['FREQUENCY'] = np.where(src_df['PRODUCT_GRP']=='Prepaid', 'Daily', 'Bill Cycle')
src_df['TM_KEY_YR'] = src_df['TM_KEY_MTH'].apply(str).str[:4].astype(int)

src_df#.tail(3)

,CUST_TYPE,TM_KEY_MTH,METRIC_CD,METRIC_NAME,TARGET_MTH,COMP_CD,VERSION,AREA_TYPE,AREA_KEY,AREA_DESC,PRODUCT_GRP,FREQUENCY,TM_KEY_YR
0,B2C,202601,TB1R000100,Prepaid Revenue : TMH,3.040760e+07,TRUE,T,HH,907030,"BKK : Bang Khen, Lat Phrao, Wang Thonglang",Prepaid,Daily,2026
1,B2C,202601,TB1R000100,Prepaid Revenue : TMH,2.079064e+07,TRUE,T,HH,907016,"BKK : Bang Sue, Chatuchak",Prepaid,Daily,2026
2,B2C,202601,TB1R000100,Prepaid Revenue : TMH,2.664091e+07,TRUE,T,HH,907017,"BKK : Don Mueang, Sai Mai, Lak Si",Prepaid,Daily,2026
3,B2C,202601,TB1R000100,Prepaid Revenue : TMH,4.288086e+07,TRUE,T,HH,907019,"BKK : Lat Krabang, Nong Chok, Khlong Sam Wa",Prepaid,Daily,2026
4,B2C,202601,TB1R000100,Prepaid Revenue : TMH,2.587798e+07,TRUE,T,HH,907020,"BKK : Min Buri, Khan Na Yao, Bueng Kum",Prepaid,Daily,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1150,B2C,202603,TB4R000100,TVS Revenue,6.508133e+05,TRUE,T,HH,906111,TRANG,TVS,Bill Cycle,2026
1151,B2C,202603,TB4R000100,TVS Revenue,2.730801e+05,TRUE,T,HH,906112,YALA,TVS,Bill Cycle,2026
1152,B2B,202601,TB2R020100,Postpaid Revenue B2B : TMH,3.182800e+08,TRUE,T,P,P,Nationwide,Postpaid,Bill Cycle,2026
1153,B2B,202602,TB2R020100,Postpaid Revenue B2B : TMH,3.182800e+08,TRUE,T,P,P,Nationwide,Postpaid,Bill Cycle,2026


In [38]:
''' Master Data '''

# DIM_TIME
dt_file = '../../data/CFW/data/dim_time.csv'
# dt_cols = ['TM_KEY_YR', 'MONTH_SHORT', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'DAYS_IN_MONTH']
dt_cols = ['TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'DAY_NO', 'DAYS_IN_MONTH', 'PERIODFLAG']
dt_df = pd.read_csv(dt_file, usecols=dt_cols)

# DIM_MOOC_AREA
mooc_file = '../../data/CFW/data/dim_mooc_area.csv'
mooc_cols = ['ZONE_TYPE', 'TEAM_CODE', 'ORGID_G', 'TDS_SGMD', 'ORGID_R', 'TDS_RGM_CODE', 'ORGID_H', 'HOP_HINT', 'TDS_PROVINCE', 'PROVINCE_ENG', 'PROVINCE_TH', 'ORGID_HH', 'D_CLUSTER', 'CCAATT', 'REMARK']
mooc_df = pd.read_csv(mooc_file, usecols=mooc_cols)
mooc_df = mooc_df.loc[(mooc_df['REMARK']!='Dummy') & (mooc_df['TEAM_CODE']!='ไม่ระบุ') & (mooc_df['HOP_HINT']!='True Corp')]

# Create HH level
mooc_hh_df = mooc_df[['ZONE_TYPE', 'ORGID_G', 'TDS_SGMD', 'ORGID_H', 'HOP_HINT', 'ORGID_HH', 'D_CLUSTER']].drop_duplicates()
mooc_hh_df.dropna(how='all', inplace=True)
mooc_hh_df['AREA_KEY'] = mooc_hh_df['ORGID_HH'].astype(int).astype(str)

In [39]:
''' Portion Data '''

portion_file = '../../data/CFW/data/revenue_portion.xlsx'

new_existing_sheet = 'New & Existing'
new_existing_df = pd.read_excel(portion_file, sheet_name=new_existing_sheet, index_col=None) 
new_existing_df = new_existing_df.loc[new_existing_df['TM_KEY_YR']==2026]
new_existing_cols = ['PRODUCT_GRP', 'TM_KEY_MTH', 'NEW', 'EXIST']
new_existing_df = new_existing_df[new_existing_cols]
# new_existing_df

bill_cycle_sheet = 'Bill Cycle'
bill_cycle_df = pd.read_excel(portion_file, sheet_name=bill_cycle_sheet, index_col=None) 
bill_cycle_df = bill_cycle_df.loc[bill_cycle_df['TM_KEY_YR']==2026]
bill_cycle_cols = ['METRIC_CD', 'TM_KEY_YR', 'BILLING_DAY', 'BILL_PORTION']
bill_cycle_df = bill_cycle_df[bill_cycle_cols]
# bill_cycle_df

# src_df_cols = ['TM_KEY_MTH', 'METRIC_CD', 'METRIC_NAME', 'METRIC_VALUE', 'COMP_CD', 'VERSION', 'AREA_TYPE', 'AREA_CD', 'AREA_DESC']
# src_df = src_df[src_df_cols]
# src_df.rename(columns={'AREA_CD': 'AREA_KEY', 'METRIC_VALUE': 'MTH_VALUE'}, inplace=True)

### Step 2 : Aggregate Data

In [40]:
# ''' Example DataFrame '''

# src_df.tail(3)
# dt_df.tail(3)
# mooc_df.tail(3)
# mooc_h_df.tail(3)
# mooc_h_df.loc[mooc_h_df['ORGID_H'].str.contains('^0')].tail(3)

In [41]:
''' Filter Rawdata

    TB1R000100	Prepaid Revenue : TMH
    TB2R010100	Postpaid Revenue B2C : TMH
    TB2R020100	Postpaid Revenue B2B : TMH
    TB3R000100	TOL Revenue
    TB4R000100	TVS Revenue
'''

''' Filter '''
raw_df = src_df.copy()
# raw_df = raw_df.query(" TM_KEY_MTH == 202603 and METRIC_CD == 'TB1R000100' ") #Prepaid Revenue : TMH
raw_df = raw_df.loc[raw_df['TM_KEY_MTH']==202603]
# raw_df = raw_df.loc[raw_df['TM_KEY_MTH']>=202601]
# raw_df = raw_df.loc[raw_df['METRIC_CD']=='TB1R000100'] #Prepaid Revenue : TMH
# raw_df = raw_df.loc[raw_df['TM_KEY_MTH'].isin([202501, 202502, 202503])]
# raw_df = raw_df.loc[raw_df['METRIC_CD'].isin(['TB1R000100', 'TB2R010100', 'TB3R000100', 'TB4R000100'])]

''' Data Test '''
# raw_df = raw_df.loc[raw_df['TM_KEY_MTH']==202504]
# raw_df = raw_df.loc[raw_df['METRIC_CD']=='TB1R000100']
# raw_df = raw_df.loc[raw_df['AREA_KEY']=='902033']
# raw_df = raw_df.loc[raw_df['AREA_KEY'].isna()]

raw_df = raw_df.reset_index(drop=True)
print(f'\nraw_df : {raw_df.shape[0]} rows, {raw_df.shape[1]} columns')
raw_df#.tail(3)


raw_df : 1155 rows, 13 columns


,CUST_TYPE,TM_KEY_MTH,METRIC_CD,METRIC_NAME,TARGET_MTH,COMP_CD,VERSION,AREA_TYPE,AREA_KEY,AREA_DESC,PRODUCT_GRP,FREQUENCY,TM_KEY_YR
0,B2C,202601,TB1R000100,Prepaid Revenue : TMH,3.040760e+07,TRUE,T,HH,907030,"BKK : Bang Khen, Lat Phrao, Wang Thonglang",Prepaid,Daily,2026
1,B2C,202601,TB1R000100,Prepaid Revenue : TMH,2.079064e+07,TRUE,T,HH,907016,"BKK : Bang Sue, Chatuchak",Prepaid,Daily,2026
2,B2C,202601,TB1R000100,Prepaid Revenue : TMH,2.664091e+07,TRUE,T,HH,907017,"BKK : Don Mueang, Sai Mai, Lak Si",Prepaid,Daily,2026
3,B2C,202601,TB1R000100,Prepaid Revenue : TMH,4.288086e+07,TRUE,T,HH,907019,"BKK : Lat Krabang, Nong Chok, Khlong Sam Wa",Prepaid,Daily,2026
4,B2C,202601,TB1R000100,Prepaid Revenue : TMH,2.587798e+07,TRUE,T,HH,907020,"BKK : Min Buri, Khan Na Yao, Bueng Kum",Prepaid,Daily,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1150,B2C,202603,TB4R000100,TVS Revenue,6.508133e+05,TRUE,T,HH,906111,TRANG,TVS,Bill Cycle,2026
1151,B2C,202603,TB4R000100,TVS Revenue,2.730801e+05,TRUE,T,HH,906112,YALA,TVS,Bill Cycle,2026
1152,B2B,202601,TB2R020100,Postpaid Revenue B2B : TMH,3.182800e+08,TRUE,T,P,P,Nationwide,Postpaid,Bill Cycle,2026
1153,B2B,202602,TB2R020100,Postpaid Revenue B2B : TMH,3.182800e+08,TRUE,T,P,P,Nationwide,Postpaid,Bill Cycle,2026


In [42]:
''' Join New & Existing '''

merge_new_existing_df = pd.merge(raw_df, new_existing_df, how='left', on=['PRODUCT_GRP', 'TM_KEY_MTH'])
merge_new_existing_df['TARGET_MTH_NEW'] = merge_new_existing_df['TARGET_MTH'] * merge_new_existing_df['NEW']
merge_new_existing_df['TARGET_MTH_EXIST'] = merge_new_existing_df['TARGET_MTH'] * merge_new_existing_df['EXIST']
merge_new_existing_df = merge_new_existing_df[['CUST_TYPE', 'TM_KEY_YR', 'TM_KEY_MTH', 'PRODUCT_GRP', 'FREQUENCY', 'METRIC_CD', 'METRIC_NAME', 'COMP_CD', 'VERSION', 'AREA_TYPE', 'AREA_KEY', 'AREA_DESC', 'TARGET_MTH', 'TARGET_MTH_NEW', 'TARGET_MTH_EXIST']]
merge_new_existing_df.tail(3)

,CUST_TYPE,TM_KEY_YR,TM_KEY_MTH,PRODUCT_GRP,FREQUENCY,METRIC_CD,METRIC_NAME,COMP_CD,VERSION,AREA_TYPE,AREA_KEY,AREA_DESC,TARGET_MTH,TARGET_MTH_NEW,TARGET_MTH_EXIST
1152,B2B,2026,202601,Postpaid,Bill Cycle,TB2R020100,Postpaid Revenue B2B : TMH,TRUE,T,P,P,Nationwide,318280000.0,1.287551e+06,3.169685e+08
1153,B2B,2026,202602,Postpaid,Bill Cycle,TB2R020100,Postpaid Revenue B2B : TMH,TRUE,T,P,P,Nationwide,318280000.0,6.448256e+06,3.117212e+08
1154,B2B,2026,202603,Postpaid,Bill Cycle,TB2R020100,Postpaid Revenue B2B : TMH,TRUE,T,P,P,Nationwide,318280000.0,1.355537e+07,3.045373e+08


In [43]:
''' Allocate Daily '''

day_df = merge_new_existing_df.loc[merge_new_existing_df['FREQUENCY']=='Daily']

day_df = pd.merge(day_df, dt_df, how='left', on='TM_KEY_MTH')

day_df['TARGET_DAY'] = day_df['TARGET_MTH'] / day_df['DAYS_IN_MONTH']
day_df['TARGET_DAY_NEW'] = day_df['TARGET_MTH_NEW'] / day_df['DAYS_IN_MONTH']
day_df['TARGET_DAY_EXIST'] = day_df['TARGET_MTH_EXIST'] / day_df['DAYS_IN_MONTH']

day_df = day_df[['CUST_TYPE', 'TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'DAYS_IN_MONTH', 'PERIODFLAG', 'PRODUCT_GRP', 'FREQUENCY', 'METRIC_CD', 'METRIC_NAME', 'COMP_CD', 'VERSION', 'AREA_TYPE', 'AREA_KEY', 'AREA_DESC', 'TARGET_MTH', 'TARGET_MTH_NEW', 'TARGET_MTH_EXIST', 'TARGET_DAY', 'TARGET_DAY_NEW', 'TARGET_DAY_EXIST']]
day_df.tail(3)

,CUST_TYPE,TM_KEY_YR,TM_KEY_MTH,TRUE_TM_KEY_WK,TM_KEY_DAY,DAYS_IN_MONTH,PERIODFLAG,PRODUCT_GRP,FREQUENCY,METRIC_CD,...,VERSION,AREA_TYPE,AREA_KEY,AREA_DESC,TARGET_MTH,TARGET_MTH_NEW,TARGET_MTH_EXIST,TARGET_DAY,TARGET_DAY_NEW,TARGET_DAY_EXIST
8637,B2C,2026,202603,2026013,20260329,31,N,Prepaid,Daily,TB1R000100,...,T,HH,906112,YALA,8.352852e+06,1.203871e+06,7.148981e+06,269446.83634,38834.532952,230612.303388
8638,B2C,2026,202603,2026014,20260330,31,N,Prepaid,Daily,TB1R000100,...,T,HH,906112,YALA,8.352852e+06,1.203871e+06,7.148981e+06,269446.83634,38834.532952,230612.303388
8639,B2C,2026,202603,2026014,20260331,31,EMQ,Prepaid,Daily,TB1R000100,...,T,HH,906112,YALA,8.352852e+06,1.203871e+06,7.148981e+06,269446.83634,38834.532952,230612.303388


In [44]:
''' Allocate Bill Cycle '''

bill_df = merge_new_existing_df.loc[merge_new_existing_df['FREQUENCY']=='Bill Cycle']
bill_df = pd.merge(bill_df, bill_cycle_df, how='left', on=['METRIC_CD', 'TM_KEY_YR'])

day_in_month_df = dt_df[['TM_KEY_MTH', 'DAYS_IN_MONTH']].drop_duplicates()
bill_df = pd.merge(bill_df, day_in_month_df, how='left', on='TM_KEY_MTH')
bill_df['BILLING_DAY'] = np.where(bill_df['BILLING_DAY']==1, bill_df['DAYS_IN_MONTH'], bill_df['BILLING_DAY'])

period_flag_df = dt_df[['TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'DAY_NO', 'PERIODFLAG']].drop_duplicates()
period_flag_df.rename(columns={'DAY_NO': 'BILLING_DAY'}, inplace=True)
bill_df = pd.merge(bill_df, period_flag_df, how='left', on=['TM_KEY_MTH', 'BILLING_DAY'])

bill_df['TARGET_DAY'] = bill_df['TARGET_MTH'] * bill_df['BILL_PORTION']
bill_df['TARGET_DAY_NEW'] = bill_df['TARGET_MTH_NEW'] * bill_df['BILL_PORTION']
bill_df['TARGET_DAY_EXIST'] = bill_df['TARGET_MTH_EXIST'] * bill_df['BILL_PORTION']

bill_df = bill_df[['CUST_TYPE', 'TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'DAYS_IN_MONTH', 'PERIODFLAG', 'PRODUCT_GRP', 'FREQUENCY', 'METRIC_CD', 'METRIC_NAME', 'COMP_CD', 'VERSION', 'AREA_TYPE', 'AREA_KEY', 'AREA_DESC', 'TARGET_MTH', 'TARGET_MTH_NEW', 'TARGET_MTH_EXIST', 'TARGET_DAY', 'TARGET_DAY_NEW', 'TARGET_DAY_EXIST']]

''' Test '''
# bill_df = bill_df.loc[bill_df['PRODUCT_GRP']=='TOL']

bill_df.tail(3)

,CUST_TYPE,TM_KEY_YR,TM_KEY_MTH,TRUE_TM_KEY_WK,TM_KEY_DAY,DAYS_IN_MONTH,PERIODFLAG,PRODUCT_GRP,FREQUENCY,METRIC_CD,...,VERSION,AREA_TYPE,AREA_KEY,AREA_DESC,TARGET_MTH,TARGET_MTH_NEW,TARGET_MTH_EXIST,TARGET_DAY,TARGET_DAY_NEW,TARGET_DAY_EXIST
8667,B2B,2026,202603,2026012,20260322,31,N,Postpaid,Bill Cycle,TB2R020100,...,T,P,P,Nationwide,318280000.0,1.355537e+07,3.045373e+08,1.528738e+07,6.510811e+05,1.462730e+07
8668,B2B,2026,202603,2026013,20260325,31,N,Postpaid,Bill Cycle,TB2R020100,...,T,P,P,Nationwide,318280000.0,1.355537e+07,3.045373e+08,3.543563e+07,1.509184e+06,3.390558e+07
8669,B2B,2026,202603,2026013,20260328,31,N,Postpaid,Bill Cycle,TB2R020100,...,T,P,P,Nationwide,318280000.0,1.355537e+07,3.045373e+08,4.009582e+07,1.707659e+06,3.836456e+07


In [45]:
''' Concat Day & Bill '''

day_and_bill_df = pd.concat([day_df, bill_df])

''' Test '''
# day_and_bill_df = day_and_bill_df.loc[day_and_bill_df['PRODUCT_GRP']=='Postpaid']
# day_and_bill_df = day_and_bill_df.loc[day_and_bill_df['AREA_KEY'].isin(['P', '902033'])]

print(f'day_and_bill_df : {day_and_bill_df.shape[0]} rows, {day_and_bill_df.shape[1]} columns')
day_and_bill_df.tail(3)

day_and_bill_df : 17310 rows, 22 columns


,CUST_TYPE,TM_KEY_YR,TM_KEY_MTH,TRUE_TM_KEY_WK,TM_KEY_DAY,DAYS_IN_MONTH,PERIODFLAG,PRODUCT_GRP,FREQUENCY,METRIC_CD,...,VERSION,AREA_TYPE,AREA_KEY,AREA_DESC,TARGET_MTH,TARGET_MTH_NEW,TARGET_MTH_EXIST,TARGET_DAY,TARGET_DAY_NEW,TARGET_DAY_EXIST
8667,B2B,2026,202603,2026012,20260322,31,N,Postpaid,Bill Cycle,TB2R020100,...,T,P,P,Nationwide,318280000.0,1.355537e+07,3.045373e+08,1.528738e+07,6.510811e+05,1.462730e+07
8668,B2B,2026,202603,2026013,20260325,31,N,Postpaid,Bill Cycle,TB2R020100,...,T,P,P,Nationwide,318280000.0,1.355537e+07,3.045373e+08,3.543563e+07,1.509184e+06,3.390558e+07
8669,B2B,2026,202603,2026013,20260328,31,N,Postpaid,Bill Cycle,TB2R020100,...,T,P,P,Nationwide,318280000.0,1.355537e+07,3.045373e+08,4.009582e+07,1.707659e+06,3.836456e+07


In [46]:
''' Join Area '''

merge_hh_df = pd.merge(day_and_bill_df, mooc_hh_df, how='left', on='AREA_KEY')
# merge_hh_df = pd.merge(day_and_bill_df.loc[day_and_bill_df['AREA_TYPE']=='HH'], mooc_hh_df, how='left', on='AREA_KEY')

''' Test '''
# merge_hh_df = merge_hh_df.loc[merge_hh_df['PRODUCT_GRP']=='TOL']
# merge_hh_df = merge_hh_df.loc[merge_hh_df['AREA_KEY'].isin(['P', '902033'])]
# merge_hh_df = merge_hh_df.loc[merge_hh_df['TM_KEY_DAY']==20250302]

merge_hh_df = merge_hh_df[['CUST_TYPE', 'TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'DAYS_IN_MONTH', 'PERIODFLAG', 'PRODUCT_GRP', 'FREQUENCY', 'METRIC_CD', 'METRIC_NAME', 'COMP_CD', 'VERSION', 'AREA_TYPE', 'AREA_KEY', 'AREA_DESC', 'ZONE_TYPE', 'ORGID_G', 'TDS_SGMD', 'ORGID_H', 'HOP_HINT', 'ORGID_HH', 'D_CLUSTER', 'TARGET_MTH', 'TARGET_MTH_NEW', 'TARGET_MTH_EXIST', 'TARGET_DAY', 'TARGET_DAY_NEW', 'TARGET_DAY_EXIST']]
print(f'merge_hh_df : {merge_hh_df.shape[0]} rows, {merge_hh_df.shape[1]} columns')
merge_hh_df.tail(3)

merge_hh_df : 17310 rows, 29 columns


,CUST_TYPE,TM_KEY_YR,TM_KEY_MTH,TRUE_TM_KEY_WK,TM_KEY_DAY,DAYS_IN_MONTH,PERIODFLAG,PRODUCT_GRP,FREQUENCY,METRIC_CD,...,ORGID_H,HOP_HINT,ORGID_HH,D_CLUSTER,TARGET_MTH,TARGET_MTH_NEW,TARGET_MTH_EXIST,TARGET_DAY,TARGET_DAY_NEW,TARGET_DAY_EXIST
17307,B2B,2026,202603,2026012,20260322,31,N,Postpaid,Bill Cycle,TB2R020100,...,NaN,NaN,NaN,NaN,318280000.0,1.355537e+07,3.045373e+08,1.528738e+07,6.510811e+05,1.462730e+07
17308,B2B,2026,202603,2026013,20260325,31,N,Postpaid,Bill Cycle,TB2R020100,...,NaN,NaN,NaN,NaN,318280000.0,1.355537e+07,3.045373e+08,3.543563e+07,1.509184e+06,3.390558e+07
17309,B2B,2026,202603,2026013,20260328,31,N,Postpaid,Bill Cycle,TB2R020100,...,NaN,NaN,NaN,NaN,318280000.0,1.355537e+07,3.045373e+08,4.009582e+07,1.707659e+06,3.836456e+07


In [47]:
''' Aggregate P, G, H, HH '''

# agg_cols = ['TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'METRIC_CD', 'METRIC_NAME', 'COMP_CD', 'VERSION', 'AREA_NO', 'AREA_TYPE', 'AREA_CD', 'AREA_NAME', 'DAY_VALUE', 'MTH_VALUE'] # , 'FREQUENCY', 'REMARK'
agg_cols = ['TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'METRIC_CD', 'METRIC_NAME', 'COMP_CD', 'VERSION', 'AREA_NO', 'AREA_TYPE', 'AREA_CD', 'AREA_NAME', 'FREQUENCY', 'TARGET_MTH', 'TARGET_MTH_NEW', 'TARGET_MTH_EXIST', 'TARGET_DAY', 'TARGET_DAY_NEW', 'TARGET_DAY_EXIST']

# P : Nationwide
agg_p_df = merge_hh_df.groupby(['TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'METRIC_CD', 'METRIC_NAME', 'COMP_CD', 'VERSION', 'FREQUENCY']).agg({'TARGET_MTH': 'sum', 'TARGET_MTH_NEW': 'sum', 'TARGET_MTH_EXIST': 'sum', 'TARGET_DAY': 'sum', 'TARGET_DAY_NEW': 'sum', 'TARGET_DAY_EXIST': 'sum'}).reset_index()
agg_p_df['AREA_NO'] = 1
agg_p_df['AREA_TYPE'] = 'P'
agg_p_df['AREA_CD'] = 'P'
agg_p_df['AREA_NAME'] = 'Nationwide'
agg_p_df = agg_p_df.loc[:, agg_cols]
# agg_p_df[agg_p_df['TM_KEY_DAY']==20240501]

# G : Region
agg_g_df = merge_hh_df.groupby(['TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'METRIC_CD', 'METRIC_NAME', 'COMP_CD', 'VERSION', 'FREQUENCY', 'ORGID_G', 'TDS_SGMD']).agg({'TARGET_MTH': 'sum', 'TARGET_MTH_NEW': 'sum', 'TARGET_MTH_EXIST': 'sum', 'TARGET_DAY': 'sum', 'TARGET_DAY_NEW': 'sum', 'TARGET_DAY_EXIST': 'sum'}).reset_index()
agg_g_df['AREA_NO'] = 2
agg_g_df['AREA_TYPE'] = 'G'
agg_g_df.rename(columns={'ORGID_G': 'AREA_CD'}, inplace=True)
agg_g_df.rename(columns={'TDS_SGMD': 'AREA_NAME'}, inplace=True)
agg_g_df = agg_g_df.loc[:, agg_cols]
# agg_g_df[agg_g_df['TM_KEY_DAY']==20240501]

# H : HOP_HINT
agg_h_df = merge_hh_df.groupby(['TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'METRIC_CD', 'METRIC_NAME', 'COMP_CD', 'VERSION', 'FREQUENCY', 'ORGID_H', 'HOP_HINT']).agg({'TARGET_MTH': 'sum', 'TARGET_MTH_NEW': 'sum', 'TARGET_MTH_EXIST': 'sum', 'TARGET_DAY': 'sum', 'TARGET_DAY_NEW': 'sum', 'TARGET_DAY_EXIST': 'sum'}).reset_index()
agg_h_df['AREA_NO'] = 3
agg_h_df['AREA_TYPE'] = 'H'
agg_h_df.rename(columns={'ORGID_H': 'AREA_CD'}, inplace=True)
agg_h_df.rename(columns={'HOP_HINT': 'AREA_NAME'}, inplace=True)
agg_h_df = agg_h_df.loc[:, agg_cols]
# agg_h_df[agg_h_df['TM_KEY_DAY']==20240501]

# HH : D_CLUSTER
agg_hh_df = merge_hh_df.groupby(['TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'METRIC_CD', 'METRIC_NAME', 'COMP_CD', 'VERSION', 'FREQUENCY', 'ORGID_HH', 'D_CLUSTER']).agg({'TARGET_MTH': 'sum', 'TARGET_MTH_NEW': 'sum', 'TARGET_MTH_EXIST': 'sum', 'TARGET_DAY': 'sum', 'TARGET_DAY_NEW': 'sum', 'TARGET_DAY_EXIST': 'sum'}).reset_index()
agg_hh_df['AREA_NO'] = 4
agg_hh_df['AREA_TYPE'] = 'HH'
agg_hh_df['ORGID_HH'] = agg_hh_df['ORGID_HH'].astype(int).astype(str)
agg_hh_df.rename(columns={'ORGID_HH': 'AREA_CD'}, inplace=True)
agg_hh_df.rename(columns={'D_CLUSTER': 'AREA_NAME'}, inplace=True)
agg_hh_df = agg_hh_df.loc[:, agg_cols]
# agg_hh_df[agg_hh_df['TM_KEY_DAY']==20240601]

# Concat DataFrame
agg_all_area_df = pd.concat([agg_p_df, agg_g_df, agg_h_df, agg_hh_df], ignore_index=True)
# agg_all_area_df['AGG_TYPE'] = 'S'
# agg_all_area_df['FREQUENCY'] = np.where(agg_all_area_df['FREQUENCY']=='DAY', 'Daily', 'Bill Cycle')
# agg_all_area_df['REMARK'] = 'Allocate from 96 Cluster (HH level)'
# # agg_all_area_df['REMARK'] = agg_all_area_df['TM_KEY_MTH'].apply(lambda x: 'H Level 64 Province' if x>=202401 and x<=202403 else 'HH Level 96 Cluster')
# agg_all_area_df = agg_all_area_df[['TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'METRIC_CD', 'METRIC_NAME', 'COMP_CD', 'VERSION', 'AREA_NO', 'AREA_TYPE', 'AREA_CD', 'AREA_NAME', 'TARGET_DAY', 'TARGET_DAY_NEW', 'TARGET_DAY_EXIST', 'TARGET_MTH', 'TARGET_MTH_NEW', 'TARGET_MTH_EXIST', 'AGG_TYPE', 'FREQUENCY', 'REMARK']]

print(f'agg_all_area_df : {agg_all_area_df.shape[0]} rows, {agg_all_area_df.shape[1]} columns')
# agg_all_area_df.loc[agg_all_area_df['TM_KEY_DAY']==20240601]
agg_all_area_df.tail(3)

agg_all_area_df : 29942 rows, 19 columns


,TM_KEY_YR,TM_KEY_MTH,TRUE_TM_KEY_WK,TM_KEY_DAY,METRIC_CD,METRIC_NAME,COMP_CD,VERSION,AREA_NO,AREA_TYPE,AREA_CD,AREA_NAME,FREQUENCY,TARGET_MTH,TARGET_MTH_NEW,TARGET_MTH_EXIST,TARGET_DAY,TARGET_DAY_NEW,TARGET_DAY_EXIST
29939,2026,202603,2026014,20260331,TB4R000100,TVS Revenue,TRUE,T,4,HH,910096,SURIN,Bill Cycle,1.325767e+06,3530.992674,1.322236e+06,368803.512750,982.256229,367821.256521
29940,2026,202603,2026014,20260331,TB4R000100,TVS Revenue,TRUE,T,4,HH,910097,UBON RATCHATHANI,Bill Cycle,1.866538e+06,4971.261733,1.861567e+06,519236.078811,1382.912187,517853.166624
29941,2026,202603,2026014,20260331,TB4R000100,TVS Revenue,TRUE,T,4,HH,910098,YASOTHON,Bill Cycle,4.547379e+05,1211.130266,4.535267e+05,126499.581827,336.913825,126162.668002


In [48]:
''' Aggregate TB2R000100 : Postpaid Revenue : TMH (Nationwide Only)'''

post_revenue_tmh_df = agg_all_area_df.loc[(agg_all_area_df['METRIC_CD'].isin(['TB2R010100', 'TB2R020100'])) & (agg_all_area_df['AREA_CD']=='P')]
post_revenue_tmh_df = post_revenue_tmh_df\
    .groupby(['TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'COMP_CD', 'VERSION', 'AREA_NO', 'AREA_TYPE', 'AREA_CD', 'AREA_NAME', 'FREQUENCY'])\
        .agg({'TARGET_MTH': 'sum', 'TARGET_MTH_NEW': 'sum', 'TARGET_MTH_EXIST': 'sum', 'TARGET_DAY': 'sum', 'TARGET_DAY_NEW': 'sum', 'TARGET_DAY_EXIST': 'sum'}).reset_index()
post_revenue_tmh_df['METRIC_CD'] = 'TB2R000100'
post_revenue_tmh_df['METRIC_NAME'] = 'Postpaid Revenue : TMH'
post_revenue_tmh_df = post_revenue_tmh_df[['TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'METRIC_CD', 'METRIC_NAME', 'COMP_CD', 'VERSION', 'AREA_NO', 'AREA_TYPE', 'AREA_CD', 'AREA_NAME', 'FREQUENCY', 'TARGET_MTH', 'TARGET_MTH_NEW', 'TARGET_MTH_EXIST', 'TARGET_DAY', 'TARGET_DAY_NEW', 'TARGET_DAY_EXIST']]

''' Test '''
# post_revenue_tmh_df = post_revenue_tmh_df.loc[post_revenue_tmh_df['PRODUCT_GRP']=='TOL']
# post_revenue_tmh_df = post_revenue_tmh_df.loc[post_revenue_tmh_df['AREA_KEY'].isin(['P', '902033'])]
# post_revenue_tmh_df = post_revenue_tmh_df.loc[post_revenue_tmh_df['TM_KEY_MTH']==202503]
# post_revenue_tmh_df = post_revenue_tmh_df.loc[post_revenue_tmh_df['TM_KEY_DAY']==20250302]
# post_revenue_tmh_df = post_revenue_tmh_df.loc[post_revenue_tmh_df['AREA_TYPE']=='P']

print(f'post_revenue_tmh_df : {post_revenue_tmh_df.shape[0]} rows, {post_revenue_tmh_df.shape[1]} columns')
post_revenue_tmh_df.tail(3)

post_revenue_tmh_df : 29 rows, 19 columns


,TM_KEY_YR,TM_KEY_MTH,TRUE_TM_KEY_WK,TM_KEY_DAY,METRIC_CD,METRIC_NAME,COMP_CD,VERSION,AREA_NO,AREA_TYPE,AREA_CD,AREA_NAME,FREQUENCY,TARGET_MTH,TARGET_MTH_NEW,TARGET_MTH_EXIST,TARGET_DAY,TARGET_DAY_NEW,TARGET_DAY_EXIST
26,2026,202603,2026013,20260325,TB2R000100,Postpaid Revenue : TMH,TRUE,T,1,P,P,Nationwide,Bill Cycle,3.778956e+09,1.609436e+08,3.615788e+09,3.741863e+08,1.593639e+07,3.580297e+08
27,2026,202603,2026013,20260328,TB2R000100,Postpaid Revenue : TMH,TRUE,T,1,P,P,Nationwide,Bill Cycle,3.778956e+09,1.609436e+08,3.615788e+09,3.944779e+08,1.680060e+07,3.774451e+08
28,2026,202603,2026014,20260331,TB2R000100,Postpaid Revenue : TMH,TRUE,T,1,P,P,Nationwide,Bill Cycle,3.778956e+09,1.609436e+08,3.615788e+09,2.286524e+07,9.738180e+05,2.187796e+07


In [49]:
''' Prepairing Latest Results '''

latest_concat_df = pd.concat([agg_all_area_df, post_revenue_tmh_df], ignore_index=True)
latest_concat_df.rename(columns={'TARGET_DAY': 'DAY_VALUE', 'TARGET_MTH': 'MTH_VALUE'}, inplace=True)
latest_concat_df['AGG_TYPE'] = 'S'
latest_concat_df['REMARK'] = 'Allocate from 96 Cluster (HH level)'

latest_results_df = latest_concat_df[['TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'METRIC_CD', 'METRIC_NAME', 'COMP_CD', 'VERSION', 'AREA_NO', 'AREA_TYPE', 'AREA_CD', 'AREA_NAME', 'DAY_VALUE', 'MTH_VALUE', 'AGG_TYPE', 'FREQUENCY', 'REMARK']]
latest_results_df.tail(3)

,TM_KEY_YR,TM_KEY_MTH,TRUE_TM_KEY_WK,TM_KEY_DAY,METRIC_CD,METRIC_NAME,COMP_CD,VERSION,AREA_NO,AREA_TYPE,AREA_CD,AREA_NAME,DAY_VALUE,MTH_VALUE,AGG_TYPE,FREQUENCY,REMARK
29968,2026,202603,2026013,20260325,TB2R000100,Postpaid Revenue : TMH,TRUE,T,1,P,P,Nationwide,3.741863e+08,3.778956e+09,S,Bill Cycle,Allocate from 96 Cluster (HH level)
29969,2026,202603,2026013,20260328,TB2R000100,Postpaid Revenue : TMH,TRUE,T,1,P,P,Nationwide,3.944779e+08,3.778956e+09,S,Bill Cycle,Allocate from 96 Cluster (HH level)
29970,2026,202603,2026014,20260331,TB2R000100,Postpaid Revenue : TMH,TRUE,T,1,P,P,Nationwide,2.286524e+07,3.778956e+09,S,Bill Cycle,Allocate from 96 Cluster (HH level)


In [50]:
sample_daily_df = latest_results_df.loc[latest_results_df['TM_KEY_DAY']==20260302]

sample_daily_df = sample_daily_df\
    .groupby(['TM_KEY_YR', 'TM_KEY_MTH', 'TRUE_TM_KEY_WK', 'TM_KEY_DAY', 'FREQUENCY', 'METRIC_CD', 'METRIC_NAME', 'AREA_TYPE'])\
        .agg({'MTH_VALUE': 'sum'}).reset_index()

mod_col_list = sample_daily_df.iloc[:, 8:].columns.tolist()
for col in mod_col_list:
    sample_daily_df[col] = sample_daily_df[col].apply(lambda x: format(x, ',.0f'))

sample_daily_df

,TM_KEY_YR,TM_KEY_MTH,TRUE_TM_KEY_WK,TM_KEY_DAY,FREQUENCY,METRIC_CD,METRIC_NAME,AREA_TYPE,MTH_VALUE
0,2026,202603,2026010,20260302,Bill Cycle,TB2R000100,Postpaid Revenue : TMH,P,"3,778,956,006"
1,2026,202603,2026010,20260302,Bill Cycle,TB2R010100,Postpaid Revenue B2C : TMH,G,"3,460,676,006"
2,2026,202603,2026010,20260302,Bill Cycle,TB2R010100,Postpaid Revenue B2C : TMH,H,"3,460,676,006"
3,2026,202603,2026010,20260302,Bill Cycle,TB2R010100,Postpaid Revenue B2C : TMH,HH,"3,460,676,006"
4,2026,202603,2026010,20260302,Bill Cycle,TB2R010100,Postpaid Revenue B2C : TMH,P,"3,460,676,006"
5,2026,202603,2026010,20260302,Bill Cycle,TB2R020100,Postpaid Revenue B2B : TMH,P,"318,280,000"
6,2026,202603,2026010,20260302,Bill Cycle,TB3R000100,TOL Revenue,G,"1,629,206,095"
7,2026,202603,2026010,20260302,Bill Cycle,TB3R000100,TOL Revenue,H,"1,629,206,095"
8,2026,202603,2026010,20260302,Bill Cycle,TB3R000100,TOL Revenue,HH,"1,629,206,095"
9,2026,202603,2026010,20260302,Bill Cycle,TB3R000100,TOL Revenue,P,"1,629,206,095"


### Step 3 : Insert to "INTERIM_VINSIGHT_DATA"
    Delete -> Insert

In [51]:
''' Input Parameter '''

# Create list
month_list = latest_results_df['TM_KEY_MTH'].unique().tolist()
mt_cd_list = latest_results_df['METRIC_CD'].unique().tolist()
# mt_cd_list = [
# 'TB1R000100' #Prepaid Revenue : TMH
# , 'TB2R010100' #Postpaid Revenue B2C : TMH
# , 'TB3R000100' #TOL Revenue
# , 'TB4R000100' #TVS Revenue
# ]

if len(mt_cd_list) == 1:
    mt_cd_list = str(mt_cd_list).replace(r'[', '(').replace(r']', ')')
else:
    mt_cd_list = tuple(mt_cd_list)

# Create Param
# v_param = dict(mth_start=202406, mth_end=202408, metric_cd=mt_cd_list)
v_param = dict(mth_start=min(month_list), mth_end=max(month_list), metric_cd=mt_cd_list)
v_target_schema = 'AUTOKPI'
v_target_table = 'INTERIM_VINSIGHT_DATA'

# query_delete = f"DELETE {v_target_schema}.{v_target_table} WHERE TM_KEY_MTH BETWEEN {v_param['mth_start']} AND {v_param['mth_end']} AND METRIC_CD IN {v_param['metric_cd']}"
query_delete = f"""
    DELETE {v_target_schema}.{v_target_table} 
    WHERE VERSION = 'T'
    AND METRIC_CD IN {v_param['metric_cd']}
    AND TM_KEY_MTH BETWEEN {v_param['mth_start']} AND {v_param['mth_end']} 
"""

print(f"\nParameter...\n\n   -> TM_KEY_MTH BETWEEN {v_param['mth_start']} AND {v_param['mth_end']}\n   -> METRIC_CD IN {v_param['metric_cd']}")
print(f'\nDataFrame...\n\n   -> latest_results_df : {latest_results_df.shape[0]} rows, {latest_results_df.shape[1]} columns') 
print(f'\nquery_delete...\n{query_delete}')


Parameter...

   -> TM_KEY_MTH BETWEEN 202601 AND 202603
   -> METRIC_CD IN ('TB1R000100', 'TB2R010100', 'TB2R020100', 'TB3R000100', 'TB4R000100', 'TB2R000100')

DataFrame...

   -> latest_results_df : 29971 rows, 17 columns

query_delete...

    DELETE AUTOKPI.INTERIM_VINSIGHT_DATA 
    WHERE VERSION = 'T'
    AND METRIC_CD IN ('TB1R000100', 'TB2R010100', 'TB2R020100', 'TB3R000100', 'TB4R000100', 'TB2R000100')
    AND TM_KEY_MTH BETWEEN 202601 AND 202603 



In [52]:
''' DELETE -> INSERT '''

job_start_datetime = dt.datetime.now().strftime('%Y-%m-%d, %H:%M:%S')
print(f'\nJob Start... {job_start_datetime}')


# Create rows from DataFrame
rows = [tuple(x) for x in latest_results_df.values]


# Connect : AKPIPRD
dsn = f'{AKPIPRD_user}/{AKPIPRD_pwd}@{AKPIPRD_host}:{AKPIPRD_port}/{AKPIPRD_db}'
conn = oracledb.connect(dsn)
print(f'\n{AKPIPRD_db} : Connected')
cur = conn.cursor()
print(f'\nProcessing...')


try:
    # # Truncate
    # cur.execute(f"TRUNCATE TABLE {v_target_schema}.{v_target_table}")
    # print(f'\n   -> TRUNCATE : "{v_target_table}" : Done !')

    # Delete
    cur.execute(query_delete)
    print(f'\n   -> DELETE : "{v_target_table}" : Done !')
    
    # Insert
    cur.executemany(f"""
        INSERT INTO {v_target_table} 
        (TM_KEY_YR, TM_KEY_MTH, TRUE_TM_KEY_WK, TM_KEY_DAY, METRIC_CD, METRIC_NAME, COMP_CD, VERSION, AREA_NO, AREA_TYPE, AREA_CD, AREA_NAME, DAY_VALUE, MTH_VALUE, AGG_TYPE, FREQUENCY, REMARK) 
        VALUES (:1,:2,:3,:4,:5,:6,:7,:8,:9,:10,:11,:12,:13,:14,:15,:16,:17)
        """, rows)
    print(f'\n   -> INSERT : "{v_target_table}" : Done !')

    cur.close()
    conn.commit()


except oracledb.DatabaseError as e:
    print(f'\nError with Oracle : {e}')


finally:
    conn.close()
    print(f'\n{AKPIPRD_db} : Disconnected')
    print(f'\nJob Done !!!')



Job Start... 2026-03-19, 11:38:36



AKPIPRD : Connected

Processing...

   -> DELETE : "INTERIM_VINSIGHT_DATA" : Done !

   -> INSERT : "INTERIM_VINSIGHT_DATA" : Done !

AKPIPRD : Disconnected

Job Done !!!


### Step 4 : Check Result "INTERIM_VINSIGHT_DATA"

In [53]:
''' Create Result DataFrame '''

# Connect : AKPIPRD
tgt_dsn = f'{AKPIPRD_user}/{AKPIPRD_pwd}@{AKPIPRD_host}:{AKPIPRD_port}/{AKPIPRD_db}'
tgt_conn = oracledb.connect(tgt_dsn)
tgt_cur = tgt_conn.cursor()


try:
    # Get : Result Data Summary
    tgt_cur.execute("""
        SELECT TM_KEY_MTH, METRIC_CD, METRIC_NAME, COMP_CD, VERSION
            , SUM(CASE WHEN AREA_TYPE = 'P' THEN DAY_VALUE END) AS P
            , SUM(CASE WHEN AREA_TYPE = 'G' THEN DAY_VALUE END) AS G
            , SUM(CASE WHEN AREA_TYPE = 'H' THEN DAY_VALUE END) AS H
            , SUM(CASE WHEN AREA_TYPE = 'HH' THEN DAY_VALUE END) AS HH
            , MAX(LOAD_DATE) LOAD_DATE
        FROM AUTOKPI.INTERIM_VINSIGHT_DATA
        WHERE VERSION = 'T'
        AND METRIC_CD IN (
            'TB1R000100' --Prepaid Revenue : TMH
            , 'TB1R000101' --Prepaid New Revenue : TMH (2024 only)
            , 'TB1R000102' --Prepaid Existing Revenue : TMH (2024 only)
            , 'TB2R000100' --Postpaid Revenue : TMH
            , 'TB2R000101' --Postpaid New Revenue : TMH (2024 only)
            , 'TB2R000102' --Postpaid Existing Revenue : TMH (2024 only)
            , 'TB2R010100' --Postpaid Revenue B2C : TMH
            , 'TB2R020100' --Postpaid Revenue B2B : TMH
            , 'TB3R000100' --TOL Revenue
            , 'TB3R000101' --TOL New Revenue (2024 only)
            , 'TB3R000102' --TOL Existing Revenue (2024 only)
            , 'TB4R000100' --TVS Revenue
            , 'TB4R000101' --TVS New Revenue (2024 only)
            , 'TB4R000102' --TVS Existing Revenue (2024 only)
            )
        AND TM_KEY_MTH >= 202601
        GROUP BY TM_KEY_MTH, METRIC_CD, METRIC_NAME, COMP_CD, VERSION
        --ORDER BY TM_KEY_MTH, METRIC_CD
    """)
    rows = tgt_cur.fetchall()
    print(f'\nGet : Fact Summary...')
    chk_result_df = pd.DataFrame.from_records(rows, columns=[x[0] for x in tgt_cur.description])
    print(f'\n   -> chk_result_df : {chk_result_df.shape[0]} rows, {chk_result_df.shape[1]} columns') 
    
    # # Display
    # tmp_result_df = chk_result_df.copy()
    # # tmp_result_df = tmp_result_df.replace(np.nan, None)
    # # tmp_result_df.iloc[:, 4:18] = tmp_result_df.iloc[:, 4:18].fillna(0)
    # mod_col_list = tmp_result_df.iloc[:, 5:9].columns.tolist()
    # for col in mod_col_list:
    #     tmp_result_df[col] = tmp_result_df[col].apply(lambda x: format(x, ',.2f') if re.search('%', col) else format(x, ',.0f'))
    # print(f'\n{tmp_result_df.to_string(max_cols=10)}') #max_rows=1000

    tgt_cur.close()


except oracledb.DatabaseError as e:
    print(f'\nError with Oracle : {e}')


finally:
    tgt_conn.close()


Get : Fact Summary...

   -> chk_result_df : 18 rows, 10 columns


In [54]:
mth_df = chk_result_df.loc[chk_result_df['TM_KEY_MTH']==202601].copy()
# mth_df = mth_df.replace(np.nan, None)
mth_df.iloc[:, 5:9] = mth_df.iloc[:, 5:9].fillna(0)
mod_col_list = mth_df.iloc[:, 5:9].columns.tolist()
for col in mod_col_list:
    mth_df[col] = mth_df[col].apply(lambda x: format(x, ',.2f') if re.search('%', col) else format(x, ',.0f'))
mth_df = mth_df.sort_values(by=['TM_KEY_MTH', 'METRIC_CD']).reset_index(drop=True)
mth_df

,TM_KEY_MTH,METRIC_CD,METRIC_NAME,COMP_CD,VERSION,P,G,H,HH,LOAD_DATE
0,202601,TB1R000100,Prepaid Revenue : TMH,TRUE,T,"2,644,084,137","2,644,084,137","2,644,084,137","2,644,084,137",2026-03-19 11:38:48.673808
1,202601,TB2R000100,Postpaid Revenue : TMH,TRUE,T,"3,732,760,181",0,0,0,2026-03-19 11:38:48.673808
2,202601,TB2R010100,Postpaid Revenue B2C : TMH,TRUE,T,"3,414,480,181","3,414,480,181","3,414,480,181","3,414,480,181",2026-03-19 11:38:48.673808
3,202601,TB2R020100,Postpaid Revenue B2B : TMH,TRUE,T,"318,280,000",0,0,0,2026-03-19 11:38:48.673808
4,202601,TB3R000100,TOL Revenue,TRUE,T,"1,615,628,684","1,615,628,684","1,615,628,684","1,615,628,684",2026-03-19 11:38:48.673808
5,202601,TB4R000100,TVS Revenue,TRUE,T,"289,600,000","289,600,000","289,600,000","289,600,000",2026-03-19 11:38:48.673808


In [55]:
mth_df = chk_result_df.loc[chk_result_df['TM_KEY_MTH']==202602].copy()
# mth_df = mth_df.replace(np.nan, None)
mth_df.iloc[:, 5:9] = mth_df.iloc[:, 5:9].fillna(0)
mod_col_list = mth_df.iloc[:, 5:9].columns.tolist()
for col in mod_col_list:
    mth_df[col] = mth_df[col].apply(lambda x: format(x, ',.2f') if re.search('%', col) else format(x, ',.0f'))
mth_df = mth_df.sort_values(by=['TM_KEY_MTH', 'METRIC_CD']).reset_index(drop=True)
mth_df

,TM_KEY_MTH,METRIC_CD,METRIC_NAME,COMP_CD,VERSION,P,G,H,HH,LOAD_DATE
0,202602,TB1R000100,Prepaid Revenue : TMH,TRUE,T,"2,429,258,067","2,429,258,067","2,429,258,067","2,429,258,067",2026-03-19 11:38:48.673808
1,202602,TB2R000100,Postpaid Revenue : TMH,TRUE,T,"3,756,671,033",0,0,0,2026-03-19 11:38:48.673808
2,202602,TB2R010100,Postpaid Revenue B2C : TMH,TRUE,T,"3,438,391,033","3,438,391,033","3,438,391,033","3,438,391,033",2026-03-19 11:38:48.673808
3,202602,TB2R020100,Postpaid Revenue B2B : TMH,TRUE,T,"318,280,000",0,0,0,2026-03-19 11:38:48.673808
4,202602,TB3R000100,TOL Revenue,TRUE,T,"1,623,288,467","1,623,288,467","1,623,288,467","1,623,288,467",2026-03-19 11:38:48.673808
5,202602,TB4R000100,TVS Revenue,TRUE,T,"294,504,089","294,504,089","294,504,089","294,504,089",2026-03-19 11:38:48.673808


In [56]:
mth_df = chk_result_df.loc[chk_result_df['TM_KEY_MTH']==202603].copy()
# mth_df = mth_df.replace(np.nan, None)
mth_df.iloc[:, 5:9] = mth_df.iloc[:, 5:9].fillna(0)
mod_col_list = mth_df.iloc[:, 5:9].columns.tolist()
for col in mod_col_list:
    mth_df[col] = mth_df[col].apply(lambda x: format(x, ',.2f') if re.search('%', col) else format(x, ',.0f'))
mth_df = mth_df.sort_values(by=['TM_KEY_MTH', 'METRIC_CD']).reset_index(drop=True)
mth_df

,TM_KEY_MTH,METRIC_CD,METRIC_NAME,COMP_CD,VERSION,P,G,H,HH,LOAD_DATE
0,202603,TB1R000100,Prepaid Revenue : TMH,TRUE,T,"2,694,075,127","2,694,075,127","2,694,075,127","2,694,075,127",2026-03-19 11:38:48.673808
1,202603,TB2R000100,Postpaid Revenue : TMH,TRUE,T,"3,778,956,006",0,0,0,2026-03-19 11:38:48.673808
2,202603,TB2R010100,Postpaid Revenue B2C : TMH,TRUE,T,"3,460,676,006","3,460,676,006","3,460,676,006","3,460,676,006",2026-03-19 11:38:48.673808
3,202603,TB2R020100,Postpaid Revenue B2B : TMH,TRUE,T,"318,280,000",0,0,0,2026-03-19 11:38:48.673808
4,202603,TB3R000100,TOL Revenue,TRUE,T,"1,629,206,095","1,629,206,095","1,629,206,095","1,629,206,095",2026-03-19 11:38:48.673808
5,202603,TB4R000100,TVS Revenue,TRUE,T,"299,408,179","299,408,179","299,408,179","299,408,179",2026-03-19 11:38:48.673808
